# Cache checker — ¿me está cacheando OpenAI el prompt?

Experimento medible sobre el **prompt caching** del proveedor (la caché que da nombre a CAG,
ver `curso-master/pildoras/cag.md`). Cada respuesta de la API trae un metadato que casi nadie mira:
`usage.input_tokens_details.cached_tokens` — cuántos tokens del input NO se han vuelto a procesar.

**Las reglas del juego (OpenAI), que vamos a verificar empíricamente:**
1. Es **automático** — no se activa nada — pero solo para prompts de **≥ 1024 tokens**.
2. Se cachea por **PREFIJO exacto**, en bloques de 128 tokens: la API busca el prefijo más largo de tu
   petición que coincida con una petición reciente. No hay "diff": si cambias algo en la posición N,
   de N en adelante no hay caché (pero lo ANTERIOR a N sí).
3. Los tokens cacheados se cobran con descuento (~50% en gpt-4o-mini; hasta 75–90% en modelos más
   nuevos — mira el pricing vigente) y el prefill es más rápido.
4. La caché vive ~5–10 min desde el último uso (hasta 1h en horas valle). Es efímera: es KV cache, no BBDD.

De aquí salen las respuestas a las preguntas típicas: *¿me cachean el system?* → sí, si va primero y
no cambia. *¿Y si el system cambia un 10%?* → depende de DÓNDE: cambio al final = se cachea casi todo;
cambio al principio = no se cachea nada. Vamos a verlo.

In [7]:
# Auto-instala el SDK si este kernel no lo tiene (vale para VS Code y Colab):
import importlib.util, subprocess, sys
if importlib.util.find_spec("openai") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    importlib.invalidate_caches()

import os, time
from openai import OpenAI

try:
    from google.colab import userdata  # type: ignore
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass

assert os.environ.get("OPENAI_API_KEY"), "Falta OPENAI_API_KEY en el entorno (o en los secretos de Colab)"

client = OpenAI()
MODEL = "gpt-4o-mini"

# Precios aproximados de gpt-4o-mini por 1M de tokens de INPUT (verifica el pricing vigente):
PRICE_INPUT = 0.15      # USD / 1M tokens sin cachear
PRICE_CACHED = 0.075    # USD / 1M tokens cacheados (~50% dto)
print("cliente listo ✔")

cliente listo ✔


In [8]:
respuesta = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hola que tal?"}])
print(respuesta.choices[0].message.content)

¡Hola! Estoy bien, gracias. ¿Y tú, cómo estás?


In [13]:
respuesta2 = client.responses.create(model=MODEL, instructions="responde en plan sevillano cerrado con faltas", input="hola que tal?")
# Ver la respuesta 2 como un diccionario clave: valor
respuesta2_dict = respuesta2.model_dump()

for clave, valor in respuesta2_dict.items():
    print(f"{clave}: {valor}")

id: resp_0f38a147d9eebe0e006a939da63b2487d2bbbbc42ae00452fe
created_at: 1788059046.0
error: None
incomplete_details: None
instructions: responde en plan sevillano cerrado con faltas
metadata: {}
model: gpt-4o-mini-2024-07-18
object: response
output: [{'id': 'msg_0f38a147d9eebe0e006a939da691f087d287cda09a7caba573', 'content': [{'annotations': [], 'text': '¡Hola, mi arma! Aquí andamo, ¿y tú? ¿Qué pasa por ahí?', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'phase': None}]
parallel_tool_calls: True
temperature: 1.0
tool_choice: auto
tools: []
top_p: 1.0
background: False
completed_at: 1788059046.0
conversation: None
max_output_tokens: None
max_tool_calls: None
moderation: None
previous_response_id: None
prompt: None
prompt_cache_key: None
prompt_cache_options: None
prompt_cache_retention: in_memory
reasoning: {'context': None, 'effort': None, 'generate_summary': None, 'mode': None, 'summary': None}
safety_identifier: None
service_

## 1. Un system prompt GRANDE (sin él no hay experimento)

El cacheo solo arranca a partir de 1024 tokens, así que fabricamos un "manual de estimación"
sintético de ~2.500 tokens que hará de system prompt — el papel del conocimiento CAG.

In [2]:
reglas = "\n".join(
    f"Rule {i}: for projects of type {t}, apply a complexity factor of {1 + i % 7 * 0.1:.1f}, "
    f"never estimate below {i * 3 + 8} hours, and always include testing as {10 + i % 5}% of the total."
    for i, t in enumerate(
        ["web_saas", "mobile_app", "internal_tool", "data_pipeline", "erp_integration", "analytics"] * 15
    )
)

SYSTEM = (
    "You are a senior software estimation assistant. Follow this manual strictly.\n\n"
    "# ESTIMATION MANUAL\n" + reglas + "\n\nAnswer in Spanish, in 2 sentences maximum."
)

print(f"system prompt: ~{len(SYSTEM) // 4} tokens (estimacion burda: 1 token ≈ 4 chars)")

system prompt: ~3564 tokens (estimacion burda: 1 token ≈ 4 chars)


## 2. El medidor

Una llamada = una fila: tokens de input, cuántos venían cacheados, latencia y coste estimado del input.

In [3]:
def medir(label: str, system: str, user: str) -> dict:
    t0 = time.perf_counter()
    r = client.responses.create(model=MODEL, instructions=system, input=user)
    ms = (time.perf_counter() - t0) * 1000
    u = r.usage
    cached = u.input_tokens_details.cached_tokens
    fresh = u.input_tokens - cached
    coste = (fresh * PRICE_INPUT + cached * PRICE_CACHED) / 1_000_000
    fila = {"label": label, "input": u.input_tokens, "cached": cached,
            "pct": round(100 * cached / u.input_tokens), "ms": round(ms), "input_usd": round(coste, 6)}
    print(f"{label:34} input={fila['input']:5}  cached={cached:5} ({fila['pct']:3}%)  "
          f"{fila['ms']:5} ms  ${fila['input_usd']:.6f}")
    return fila

print("medidor listo ✔  (columna clave: cached)")

medidor listo ✔  (columna clave: cached)


## 3. Experimento A — frío vs caliente, y el user prompt NO necesita repetirse

Tres llamadas seguidas: la primera en frío, la segunda **idéntica**, la tercera con el MISMO system
pero **otra pregunta de usuario**. Predicción: la 1ª `cached=0`; la 2ª y la 3ª cachean todo el system
(el user va DESPUÉS del prefijo, así que da igual que cambie).

In [4]:
filas = []
filas.append(medir("A1 · primera vez (frio)", SYSTEM, "¿Cuántas horas para un login OAuth?"))
time.sleep(2)
filas.append(medir("A2 · identica (caliente)", SYSTEM, "¿Cuántas horas para un login OAuth?"))
time.sleep(2)
filas.append(medir("A3 · mismo system, OTRA pregunta", SYSTEM, "¿Y para un dashboard con KPIs?"))

A1 · primera vez (frio)            input= 3679  cached=    0 (  0%)   3695 ms  $0.000552
A2 · identica (caliente)           input= 3679  cached=    0 (  0%)    972 ms  $0.000552
A3 · mismo system, OTRA pregunta   input= 3679  cached=    0 (  0%)   1961 ms  $0.000552


## 4. Experimento B — ¿y si cambio el system "solo un poco"?

La pregunta del millón. Dos mutaciones del MISMO tamaño, en sitios opuestos:
- **B1**: una palabra cambiada AL PRINCIPIO del system → rompe el prefijo entero → `cached ≈ 0`.
- **B2**: una frase añadida AL FINAL del system → el prefijo anterior sigue valiendo → `cached` alto.

Moraleja de producción: lo estable (reglas, ejemplos, tools) SIEMPRE al principio;
lo variable (fecha, datos del usuario), al final.

In [ ]:
system_b1 = SYSTEM.replace("You are a senior", "You are an expert", 1)   # cambio al PRINCIPIO
system_b2 = SYSTEM + "\nToday is a regular business day."                 # cambio al FINAL

filas.append(medir("B1 · 1 palabra cambiada al INICIO", system_b1, "¿Cuántas horas para un login OAuth?"))
time.sleep(2)
filas.append(medir("B2 · 1 frase anadida al FINAL", system_b2, "¿Cuántas horas para un login OAuth?"))

## 5. La factura del experimento

El resumen con el ahorro. Con el system de ~2.5k tokens, cada llamada "caliente" se ahorra
~la mitad del coste de input y baja la latencia del prefill — multiplícalo por miles de
peticiones/día y esa es la economía de mantener el prefijo estable.

In [ ]:
total_sin_cache = sum(f["input"] for f in filas) * PRICE_INPUT / 1_000_000
total_real = sum(f["input_usd"] for f in filas)
print(f"coste input SIN cache: ${total_sin_cache:.6f}")
print(f"coste input REAL:      ${total_real:.6f}")
print(f"ahorro:                {100 * (1 - total_real / total_sin_cache):.1f}%")

## 6. Qué apuntar de aquí

- El cacheo es **por prefijo, no por diff**: no "te cachea el 90% que coincide" — te cachea desde el
  principio hasta el primer token distinto (redondeado a bloques de 128).
- Por eso el ORDEN del prompt es una decisión económica: system estable → tools → ejemplos → y lo
  variable al final. Un timestamp al principio del system es un incendio de dinero.
- `cached=0` en la primera llamada de la mañana es normal: la caché expira en minutos (es KV cache
  del proveedor, efímera — nada que ver con nuestras cachés de respuesta en Redis).
- La latencia puede bailar (red, carga del proveedor): la señal fiable es `cached_tokens`, no los ms.
- Anthropic tiene lo mismo pero **explícito** (`cache_control` en los bloques del prompt) con hasta
  90% de descuento en lecturas; Gemini lo llama *context caching*. El mecanismo (KV cache por
  prefijo) es el mismo en todos.